# 01: Data Loading & Exploratory Data Analysis (EDA)
**Project:** AI-Powered Agricultural Procurement Decision Support System  
**Module:** Multi-Target Price Forecasting  

### Notebook Objectives:
1. **Load Raw Data:** Read `all_months.csv` from the `data/` directory.
2. **Initial Inspection:** Verify shape, data types, missing values, and column structure.
3. **Calendar Audit:** Check Bikram Sambat (BS) month alignment and year-crossing sequence (Shrawan 2082 $\rightarrow$ Baishakh 2083).
4. **Data Quality Audit:** Inspect product name variants, unit mismatches, and price range anomalies ($Min \le Avg \le Max$).
5. **Supply Measure Audit:** Explore `Volume` vs `TOTAL_sources` to analyze supply origin aggregations and identify mismatched rows.

In [23]:
import os
import pandas as pd
import numpy as np

# Configure Pandas display options for full column view
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Environment setup complete.")

Environment setup complete.


In [24]:
# Load raw data

# Path setup
data_path = os.path.join('data', 'all_months.csv')

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f" Dataset successfully loaded!")
    print(f"Raw Shape: {df.shape[0]} rows, {df.shape[1]} columns")
else:
    raise FileNotFoundError(f"Dataset not found at '{data_path}'. Please ensure 'all_months.csv' exists in the 'data/' folder.")

# Preview top 5 rows
df.head()

 Dataset successfully loaded!
Raw Shape: 798 rows, 56 columns


,Product_Name,Category,bs_year,bs_month,month_idx,month_name,Volume,Min_Price,Max_Price,Avg_Price,Unit,unit_canonical,unit_changed,Total_Amount,Volume_Equals,TOTAL_sources,reconciliation_gap,import_share,n_months_present,is_balanced,Baglung,Bara,Beni,Bhutan,Birgunj,China,Dang,Dhading,Gorkha,Hetauda,India,Jhapa,Kathmandu,Kevray,Lahan,Lamjung,Local,Marpha,Mugu,Mustang,Myagdi,Naew,Narayangadh,Nawalparasi,Nuwakot,Palpa,Parbat,Rolpa,Rukum,Rupandehi,Salyan,Sarlahi,Sindhuli,Sunsari,Syangja,Tanahu
0,Akabary_Chilly,Vegetable,2082,9,6,poush,880,400.00,600.00,500.00,kg,kg,False,440000.00,True,880,0,0.00,5,False,0,0,0,0,0,0,0,0,240,0,0,0,0,0,0,0,0,0,0,0,0,0,640,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Akabary_Chilly,Vegetable,2082,10,7,magh,540,300.00,900.00,628.57,kg,kg,False,339427.80,True,540,0,0.00,5,False,0,0,0,0,0,0,0,0,540,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Akabary_Chilly,Vegetable,2082,11,8,falgun,0,300.00,800.00,555.56,kg,kg,False,0.00,True,0,0,NaN,5,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Akabary_Chilly,Vegetable,2082,12,9,chaitra,5920,300.00,500.00,439.77,kg,kg,False,2603438.40,True,5920,0,0.89,5,False,0,0,0,0,0,0,0,0,0,0,5260,0,0,0,0,0,0,0,0,0,0,0,660,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Akabary_Chilly,Vegetable,2083,1,10,baishakh,13770,400.00,500.00,450.00,kg,kg,False,6196500.00,True,13770,0,0.29,5,False,0,0,0,0,0,0,0,0,0,0,3970,0,0,0,0,1230,0,0,0,0,0,0,8570,0,0,0,0,0,0,0,0,0,0,0,0,0


In [25]:
# Data Types and Non-Null Summary
print("--- DATASET INFO ---")
print(df.info())

print("\n--- MISSING VALUE SUMMARY ---")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

--- DATASET INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 798 entries, 0 to 797
Data columns (total 56 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Product_Name        798 non-null    object 
 1   Category            798 non-null    object 
 2   bs_year             798 non-null    int64  
 3   bs_month            798 non-null    int64  
 4   month_idx           798 non-null    int64  
 5   month_name          798 non-null    object 
 6   Volume              798 non-null    int64  
 7   Min_Price           791 non-null    float64
 8   Max_Price           791 non-null    float64
 9   Avg_Price           798 non-null    float64
 10  Unit                798 non-null    object 
 11  unit_canonical      798 non-null    object 
 12  unit_changed        798 non-null    bool   
 13  Total_Amount        798 non-null    float64
 14  Volume_Equals       798 non-null    bool   
 15  TOTAL_sources       798 non-null    

In [26]:
# Month - Year Distribution
# Identify month column
month_col = 'month_name' if 'month_name' in df.columns else ('raw_sheet_name' if 'raw_sheet_name' in df.columns else 'month')

if month_col in df.columns:
    print("Distinct month entries found in raw data:")
    print(df[month_col].value_counts(dropna=False))
else:
    print("Column for month names not detected directly. Columns present:")
    print(df.columns.tolist())

Distinct month entries found in raw data:
month_name
ashwin      86
magh        85
chaitra     85
baishakh    82
falgun      82
kartik      79
shrawan     78
mangsir     78
poush       74
bhadra      69
Name: count, dtype: int64


In [27]:
# Functional classification of columns
time_cols = ['bs_year', 'bs_month', 'month_idx', 'month_name']
identity_cols = ['Product_Name', 'Category', 'Unit', 'unit_canonical']
price_target_cols = ['Min_Price', 'Max_Price', 'Avg_Price', 'Total_Amount']
supply_core_cols = ['Volume', 'TOTAL_sources', 'reconciliation_gap', 'import_share']
audit_flag_cols = ['unit_changed', 'Volume_Equals', 'n_months_present', 'is_balanced']

# Isolate district/source columns dynamically
metadata_and_flags = set(time_cols + identity_cols + price_target_cols + supply_core_cols + audit_flag_cols)
district_cols = [c for c in df.columns if c not in metadata_and_flags]
print(f" Time Columns ({len(time_cols)}): {time_cols}")
print(f" Identity Columns ({len(identity_cols)}): {identity_cols}")
print(f" Price & Target Columns ({len(price_target_cols)}): {price_target_cols}")
print(f" Supply Metrics ({len(supply_core_cols)}): {supply_core_cols}")
print(f" Audit Flags ({len(audit_flag_cols)}): {audit_flag_cols}")
print(f" District / Source Origins ({len(district_cols)}): {district_cols}")

 Time Columns (4): ['bs_year', 'bs_month', 'month_idx', 'month_name']
 Identity Columns (4): ['Product_Name', 'Category', 'Unit', 'unit_canonical']
 Price & Target Columns (4): ['Min_Price', 'Max_Price', 'Avg_Price', 'Total_Amount']
 Supply Metrics (4): ['Volume', 'TOTAL_sources', 'reconciliation_gap', 'import_share']
 Audit Flags (4): ['unit_changed', 'Volume_Equals', 'n_months_present', 'is_balanced']
 District / Source Origins (36): ['Baglung', 'Bara', 'Beni', 'Bhutan', 'Birgunj', 'China', 'Dang', 'Dhading', 'Gorkha', 'Hetauda', 'India', 'Jhapa', 'Kathmandu', 'Kevray', 'Lahan', 'Lamjung', 'Local', 'Marpha', 'Mugu', 'Mustang', 'Myagdi', 'Naew', 'Narayangadh', 'Nawalparasi', 'Nuwakot', 'Palpa', 'Parbat', 'Rolpa', 'Rukum', 'Rupandehi', 'Salyan', 'Sarlahi', 'Sindhuli', 'Sunsari', 'Syangja', 'Tanahu']


In [28]:
# Check data types and null counts
print("=== MISSING VALUE SUMMARY ===")
null_series = df.isnull().sum()
nulls_present = null_series[null_series > 0]

if not nulls_present.empty:
    for col, count in nulls_present.items():
        pct = (count / len(df)) * 100
        print(f"- {col}: {count} missing values ({pct:.2f}%)")
else:
    print("No missing values found across the dataset!")

print("\n=== DATA TYPES OVERVIEW ===")
print(df.dtypes.value_counts())

=== MISSING VALUE SUMMARY ===
- Min_Price: 7 missing values (0.88%)
- Max_Price: 7 missing values (0.88%)
- import_share: 85 missing values (10.65%)

=== DATA TYPES OVERVIEW ===
int64      43
object      5
float64     5
bool        3
Name: count, dtype: int64


In [29]:
# Audit time indexing
calendar_audit = df[time_cols].drop_duplicates().sort_values('month_idx').reset_index(drop=True)
print("=== BIKRAM SAMBAT CHRONOLOGICAL SEQUENCE ===")
display(calendar_audit)

# Row count per month index
print("\nRow counts per month_idx:")
print(df['month_idx'].value_counts().sort_index())

=== BIKRAM SAMBAT CHRONOLOGICAL SEQUENCE ===


,bs_year,bs_month,month_idx,month_name
0,2082,4,1,shrawan
1,2082,5,2,bhadra
2,2082,6,3,ashwin
3,2082,7,4,kartik
4,2082,8,5,mangsir
5,2082,9,6,poush
6,2082,10,7,magh
7,2082,11,8,falgun
8,2082,12,9,chaitra
9,2083,1,10,baishakh



Row counts per month_idx:
month_idx
1     78
2     69
3     86
4     79
5     78
6     74
7     85
8     82
9     85
10    82
Name: count, dtype: int64


In [30]:
# Product profiling
print("=== PRODUCT NAME AUDIT ===")
print(f"Total Unique Products (including NaN): {df['Product_Name'].nunique(dropna=False)}")
print("\nTop 15 Most Frequent Products:")
display(df['Product_Name'].value_counts(dropna=False).head(15))

# Category profiling
print("\n=== CATEGORY AUDIT ===")
display(df['Category'].value_counts(dropna=False))

# Unit and Unit Variation Profiling
print("\n=== UNIT AUDIT ===")
display(df['Unit'].value_counts(dropna=False))

print(f"\nProducts with Unit Variations Across Months (`unit_changed == True`): {df[df['unit_changed']]['Product_Name'].nunique()}")
if df['unit_changed'].sum() > 0:
    display(df[df['unit_changed']][['Product_Name', 'month_idx', 'Unit', 'unit_canonical']].head(10))

=== PRODUCT NAME AUDIT ===
Total Unique Products (including NaN): 97

Top 15 Most Frequent Products:


Product_Name
Apple               10
Cucumber            10
Arum                10
Avocado             10
Brinjal_Long        10
Bitter_Gourd        10
Bottle_Gourd        10
Beat_Chukunder      10
Banana_Green        10
Christophine        10
Coconut             10
Chilly_Green        10
Cauliflower_Hill    10
Cabbage_Red         10
Carrot              10
Name: count, dtype: int64


=== CATEGORY AUDIT ===


Category
Vegetable    533
Fruit        265
Name: count, dtype: int64


=== UNIT AUDIT ===


Unit
kg       707
mutha     54
nos       20
dozen     17
Name: count, dtype: int64


Products with Unit Variations Across Months (`unit_changed == True`): 6


,Product_Name,month_idx,Unit,unit_canonical
129,Brinjal_Long,1,kg,kg
130,Brinjal_Long,2,kg,kg
131,Brinjal_Long,3,kg,kg
132,Brinjal_Long,4,kg,kg
133,Brinjal_Long,5,kg,kg
134,Brinjal_Long,6,kg,kg
135,Brinjal_Long,7,kg,kg
136,Brinjal_Long,8,kg,kg
137,Brinjal_Long,9,kg,kg
138,Brinjal_Long,10,kg,kg


In [31]:
# Summary statistics for prices
print("=== PRICE SUMMARY STATISTICS ===")
display(df[price_target_cols].describe())

# Check price logic (Min <= Avg <= Max)
invalid_min_max = df[df['Min_Price'] > df['Max_Price']]
invalid_avg_max = df[df['Avg_Price'] > df['Max_Price']]
invalid_avg_min = df[df['Avg_Price'] < df['Min_Price']]

print("\n=== PRICE LOGIC CHECKS ===")
print(f"Rows where Min_Price > Max_Price: {len(invalid_min_max)}")
print(f"Rows where Avg_Price > Max_Price: {len(invalid_avg_max)}")
print(f"Rows where Avg_Price < Min_Price: {len(invalid_avg_min)}")

# Target Leakage Check: Total_Amount = Volume * Avg_Price
df['calc_total'] = df['Volume'] * df['Avg_Price']
leakage_diff = np.abs(df['Total_Amount'] - df['calc_total']).max()
print(f"\nMax difference between Total_Amount and (Volume * Avg_Price): {leakage_diff:.4f}")
print(" Note: Total_Amount is a deterministic identity (Volume x Avg_Price) and MUST NOT be used as an input feature.")

=== PRICE SUMMARY STATISTICS ===


,Min_Price,Max_Price,Avg_Price,Total_Amount
count,791.00,791.00,798.00,798.00
mean,115.82,176.50,157.65,10487726.34
std,101.50,144.60,116.01,18230112.76
min,0.00,0.00,26.36,0.00
25%,50.00,80.00,78.06,704880.00
50%,80.00,150.00,116.80,3882581.70
75%,150.00,230.00,201.30,12885554.00
max,700.00,1000.00,757.61,167477450.00



=== PRICE LOGIC CHECKS ===
Rows where Min_Price > Max_Price: 0
Rows where Avg_Price > Max_Price: 71
Rows where Avg_Price < Min_Price: 1

Max difference between Total_Amount and (Volume * Avg_Price): 150678800.0000
 Note: Total_Amount is a deterministic identity (Volume x Avg_Price) and MUST NOT be used as an input feature.


In [32]:
print("=== SUPPLY RECONCILIATION SUMMARY ===")
print(f"Total Rows: {len(df)}")
print(f"Rows where Volume == TOTAL_sources (`Volume_Equals == True`): {df['Volume_Equals'].sum()}")
print(f"Rows with Mismatched Volume (`Volume_Equals == False`): {(~df['Volume_Equals']).sum()}")

# Inspect top mismatched rows
mismatches = df[~df['Volume_Equals']][['Product_Name', 'month_idx', 'Volume', 'TOTAL_sources', 'reconciliation_gap', 'import_share']]
print("\nTop Mismatched Supply Rows:")
display(mismatches.head(10))

# Import Share Analysis
print("\n=== IMPORT SHARE STATISTICS ===")
display(df['import_share'].describe())

=== SUPPLY RECONCILIATION SUMMARY ===
Total Rows: 798
Rows where Volume == TOTAL_sources (`Volume_Equals == True`): 775
Rows with Mismatched Volume (`Volume_Equals == False`): 23

Top Mismatched Supply Rows:


,Product_Name,month_idx,Volume,TOTAL_sources,reconciliation_gap,import_share
15,Apple,2,559832,531622,28210,0.91
16,Apple,3,681942,679082,2860,0.32
69,Banana_Green,1,679640,676060,3580,0.51
71,Banana_Green,3,345265,345785,520,0.71
92,Bayar,8,16805,16850,45,1.00
120,Bottle_Gourd,6,186575,186755,180,0.34
189,Cauliflower_Hill,2,157481,90255,67226,0.47
200,Chilly_Green,3,161395,165315,3920,0.11
224,Coconut,7,80995,80955,40,1.00
318,Ginger,6,53130,112230,59100,0.00



=== IMPORT SHARE STATISTICS ===


count   713.00
mean      0.42
std       0.44
min       0.00
25%       0.00
50%       0.25
75%       0.99
max       1.00
Name: import_share, dtype: float64

In [33]:
# Panel completeness
print("=== PANEL COMPLETENESS ===")
print(f"Total Distinct Products: {df['Product_Name'].nunique()}")
print(f"Products present in ALL 10 months (`is_balanced == True`): {df[df['is_balanced']]['Product_Name'].nunique()}")
print(f"Balanced Panel Total Rows (48 products x 10 months): {df['is_balanced'].sum()}")

# Distribution of months present per product
print("\nDistribution of months present per product:")
print(df.groupby('n_months_present')['Product_Name'].nunique())

# Top 15 District / Origin Supply Contributors
print("\n=== TOP 15 SUPPLY ORIGIN SOURCES ===")
district_sums = df[district_cols].sum().sort_values(ascending=False)
display(district_sums.head(15))

=== PANEL COMPLETENESS ===
Total Distinct Products: 97
Products present in ALL 10 months (`is_balanced == True`): 48
Balanced Panel Total Rows (48 products x 10 months): 480

Distribution of months present per product:
n_months_present
1     10
2      1
3      2
4      5
5      1
6      5
7      8
8      9
9     13
10    48
Name: Product_Name, dtype: int64

=== TOP 15 SUPPLY ORIGIN SOURCES ===


India          28820317
Narayangadh    11454073
Local           8719390
Dhading         5526967
Hetauda         3553553
China           3377773
Kathmandu       2824635
Birgunj          954525
Nuwakot          922270
Tanahu           826281
Mustang          766192
Gorkha           726292
Jhapa            711115
Syangja          668110
Nawalparasi      666845
dtype: int64